### Feature Engineering

This notebook covers the **feature engineering stage** of the Aadhaar analysis.

Using district–month level Aadhaar enrolment and update data, raw activity counts are transformed into **interpretable indices and indicators** that describe how Aadhaar services operate across districts.  
The emphasis is on **behavioral patterns and consistency**, rather than raw activity volume.

### Analytical Focus

The feature engineering process is designed to capture:

1. Behavioral patterns of Aadhaar activity  
2. Consistency of service usage over time  
3. Balance between enrolment and corrective updates  

### Indices Computed

The following indices are constructed in this notebook:

1. Aadhaar Infrastructure Stress Index  
2. Update Dependency Ratio  
3. Dormancy Index  
4. Biometric Decay Score (BDS)  
5. Child Compliance Index (CCI)  

All measures are derived using **simple, explainable formulas** and aggregated at the district level to enable fair comparison across regions and time periods.  
These engineered outputs serve as **structured inputs** for the subsequent **Digital Maturity Gradient** analysis.

---

### Aadhaar Master Table

The Aadhaar master table is a **pre-processed, district–month level dataset** used as the direct input for feature engineering.

It contains **aggregated enrolment and update activity** across districts and time, with **no individual-level information**.

### Key Features in the Master Table

| Feature Name        | Description |
|--------------------|-------------|
| **state**           | State or Union Territory name |
| **district**        | District name |
| **date**            | Month of observation |
| **age_0_5**         | New enrolments for children aged 0–5 |
| **age_5_17**        | New enrolments for children aged 5–17 |
| **age_18_greater**  | New enrolments for adults (18+) |
| **demo_age_5_17**   | Demographic updates for children aged 5–17 |
| **demo_age_17_**    | Demographic updates for adults |
| **bio_age_5_17**    | Biometric updates for children aged 5–17 |
| **bio_age_17_**     | Biometric updates for adults |

This master table is **loaded and used directly** in this notebook, where enrolment and update counts are transformed into indices capturing **operational behavior, service consistency, biometric quality, and compliance patterns**.

---

##### 1. Library Imports


In [1]:
import pandas as pd
import numpy as np


##### 2. Data Loading and Column Selection
<small>The Aadhaar master table is loaded and relevant enrolment and update columns are selected for feature engineering.</small>




In [2]:
master_table = pd.read_csv("data/aadhaar_master_table.csv", parse_dates=["date"])

value_cols = [
    "age_0_5", "age_5_17", "age_18_greater",
    "demo_age_5_17", "demo_age_17_",
    "bio_age_5_17", "bio_age_17_"
]

##### 3. Removal of Aggregate State District Records
<small>Aggregate state district entries are removed to retain only valid district-level records.</small>


In [3]:
master_table = master_table[
    ~((master_table["state"] == "100000") &
      (master_table["district"] == "100000"))
]

##### 4. Record Count Verification
<small>The number of remaining records is checked after filtering.</small>


In [4]:
len(master_table)

334764

##### 5. Monthly Aggregation at District Level
<small>Activity is aggregated monthly at the state district level by summing enrolment and update counts.</small>



In [5]:
monthly_base = master_table.copy()

monthly_base["month"] = monthly_base["date"].dt.to_period("M").dt.to_timestamp()

monthly_base = (
    monthly_base
    .groupby(["state", "district", "month"], as_index=False)
    .agg({
        "age_0_5": "sum",
        "age_5_17": "sum",
        "age_18_greater": "sum",
        "demo_age_5_17": "sum",
        "demo_age_17_": "sum",
        "bio_age_5_17": "sum",
        "bio_age_17_": "sum"
    })
)


##### 6. Construction of Core Activity Variables

$$
\small
\begin{aligned}
\text{New Enrolments} &\;=\; \text{age\_0\_5} + \text{age\_5\_17} + \text{age\_18\_greater} \\
\text{Total Updates} &\;=\; \text{demo\_age\_5\_17} + \text{demo\_age\_17\_} + \text{bio\_age\_5\_17} + \text{bio\_age\_17\_} \\
\text{Total Activity} &\;=\; \text{New Enrolments} + \text{Total Updates}
\end{aligned}
$$


In [6]:
monthly_base["new_enrolments"] = (
    monthly_base["age_0_5"] +
    monthly_base["age_5_17"] +
    monthly_base["age_18_greater"]
)

monthly_base["total_updates"] = (
    monthly_base["demo_age_5_17"] +
    monthly_base["demo_age_17_"] +
    monthly_base["bio_age_5_17"] +
    monthly_base["bio_age_17_"]
)

monthly_base["total_activity"] = (
    monthly_base["new_enrolments"] +
    monthly_base["total_updates"]
)


##### 7. Aadhaar Infrastructure Stress Index

Stress is defined as a ratio capturing the balance between corrective workload (updates) and expansion workload (new enrolments) at the district level.

**Formula (monthly):**

$$
\small
\text{Stress Index} = \frac{\text{Updates}}{\text{New Enrolments} + 1}
$$

**Components:**
- **Updates** represent the total number of demographic and biometric update transactions.
- **New Enrolments** represent the total number of new Aadhaar enrolments.

**Why this metric is computed:**
- It measures operational load relative to system expansion.
- Higher values indicate correction-heavy activity, often associated with legacy systems or high churn.

**Policy relevance:**
- Supports staffing allocation decisions.
- Informs Aadhaar centre capacity planning.

Infrastructure stress is computed at a **monthly level** to capture short-term operational dynamics and is subsequently aggregated to characterize each district’s typical and worst-case system pressure.


In [7]:
### Calculate Aadhar Infrastructure stress Index
monthly_base["Update-to-Enrolment Ratio"] = (
    monthly_base["total_updates"] /
    (monthly_base["new_enrolments"] + 1)
)

##### 7.1. District Level Aggregation of Stress Index
<small>
Monthly stress values are aggregated to summarize each district’s average level, peak intensity, variability, and persistence of infrastructure stress.
</small>




In [8]:
district_stress_summary = (
    monthly_base
    .groupby(["state", "district"], as_index=False)
    .agg(
        mean_update_to_enrolment_ratio=("Update-to-Enrolment Ratio", "mean"),
        max_update_to_enrolment_ratio=("Update-to-Enrolment Ratio", "max"),
        update_to_enrolment_ratio_volatility=("Update-to-Enrolment Ratio", "std"),
        active_months=("Update-to-Enrolment Ratio", "count")
    )
)

##### 7.2. Preview of District-Level Stress Metrics
<small>
The table reports district-level summaries of the Aadhaar Infrastructure Stress Index:<br>
i. <b>mean_update_to_enrolment_ratio</b>: Average monthly stress for the district<br>
ii. <b>max_update_to_enrolment_ratio</b>: Highest observed monthly stress value<br>
iii. <b>update_to_enrolment_ratio_volatility</b>: Variability of stress over time (standard deviation)<br>
iv. <b>active_months</b>: Number of months with recorded Aadhaar activity used in aggregation
</small>


In [9]:
district_stress_summary.head(10)

,state,district,mean_update_to_enrolment_ratio,max_update_to_enrolment_ratio,update_to_enrolment_ratio_volatility,active_months
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,106.583631,259.000000,97.416098,10
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,0.550000,4.000000,1.257201,10
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,6.728472,21.666667,9.163387,10
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,149.532449,671.000000,207.028878,10
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,501.677162,1437.000000,563.351927,10
5,ANDAMAN AND NICOBAR ISLANDS,SOUTH ANDAMAN,674.567832,1922.000000,725.666597,10
6,ANDHRA PRADESH,ADILABAD,2952.720898,9603.000000,3523.162448,10
7,ANDHRA PRADESH,ALLURI SITHARAMA RAJU,1600.868172,6948.000000,2209.379973,10
8,ANDHRA PRADESH,ANAKAPALLI,835.340743,3616.000000,1143.694626,10
9,ANDHRA PRADESH,ANANTAPUR,7389.462129,29828.000000,9676.180510,10


In [10]:
district_stress_summary.to_csv("data/district_summary/district_stress_summary.csv", index=False)

##### 8. Update Dependency Ratio

The Update Dependency Ratio (UDR) captures the share of Aadhaar activity in a district that is devoted to correcting existing records rather than onboarding new residents.

**Formula (monthly):**

$$
\small
\text{Update Dependency Ratio} = \frac{\text{Updates}}{\text{Total Activity} + 1}
$$

**Components:**
- **Updates** represent the total number of demographic and biometric update transactions.
- **Total Activity** represents the sum of new enrolments and updates.

**Why this metric is computed:**
- Signals enrolment quality and downstream data instability.
- High values indicate repeated corrections and correction-heavy workflows.

**Policy relevance:**
- Identifies districts requiring enrolment process improvements.
- Supports targeted operator training and quality audits.


In [11]:
# Calculating Updates dependency ratio
monthly_base["update_dependency_ratio"] = (
    monthly_base["total_updates"] /
    (monthly_base["total_activity"] + 1)
)


##### 8.1. Preview of Update Dependency Ratio

<small>
The table shows a sample of monthly Update Dependency Ratio values along with the corresponding total updates and total activity for selected district–month combinations.
</small>


In [12]:
monthly_base[[
    "state", "district", "month",
    "total_updates", "total_activity",
    "update_dependency_ratio"
]].head()


,state,district,month,total_updates,total_activity,update_dependency_ratio
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-03-01,209.0,209.0,0.995238
1,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-04-01,184.0,184.0,0.994595
2,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-05-01,180.0,180.0,0.994475
3,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-06-01,140.0,140.0,0.992908
4,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-07-01,259.0,259.0,0.996154


##### 8.2. District-Level Aggregation of Update Dependency Ratio
<small>
Monthly Update Dependency Ratio values are aggregated to summarize each district’s average level, peak dependency, and variability over time.
</small>




In [13]:
# Aggregate Update Dependency Ratio → district level
district_dependency_summary = (
    monthly_base
    .groupby(["state", "district"], as_index=False)
    .agg(
        mean_update_dependency=("update_dependency_ratio", "mean"),
        max_update_dependency=("update_dependency_ratio", "max"),
        dependency_volatility=("update_dependency_ratio", "std")
    )
)


##### 8.3. Preview of District-Level Update Dependency Metrics

<small>
The table presents district-level summaries of the Update Dependency Ratio, showing average dependency on updates, peak dependency observed, and variability in dependency over time for each district.
</small>
<small>

i. **mean_update_dependency**: Average Update Dependency Ratio for the district<br>
ii. **max_update_dependency**: Highest observed Update Dependency Ratio<br>
iii. **dependency_volatility**: Variability in dependency over time (standard deviation)
</small>





In [14]:
district_dependency_summary.head()

,state,district,mean_update_dependency,max_update_dependency,dependency_volatility
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,0.879568,0.996154,0.309773
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,0.163333,0.800000,0.285644
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,0.376071,0.955882,0.485612
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,0.873744,0.998512,0.308890
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,0.884397,0.999305,0.311274


In [15]:
district_dependency_summary.to_csv("data/district_summary/district_dependency_summary.csv", index=False)

##### 9. Dormancy Index

The Dormancy Index captures how consistently Aadhaar infrastructure is utilized within a district over time by identifying periods of inactivity.

**Metrics used:**
- Percentage of months with zero Aadhaar activity  
- Longest continuous inactive streak  

**Why this metric is computed:**
- Identifies underutilized or inaccessible Aadhaar infrastructure.
- Distinguishes service availability issues from lack of demand.

**Policy relevance:**
- Supports outreach prioritization.
- Informs deployment of mobile enrolment and update units.

Dormancy is assessed at the **monthly level** and aggregated to summarize long-term infrastructure availability and utilization patterns for each district.


In [16]:
monthly_base["is_active"] = (monthly_base["total_activity"] > 0).astype(int)

##### 9.1. Preview of Feature-Engineered Monthly Data

<small>
The table below shows a sample of the feature-engineered monthly dataset, including derived enrolment counts, update counts, activity measures, stress and dependency ratios, and the activity status flag for each district–month.
</small>

<small>

i. <b>new_enrolments</b>: Total new Aadhaar enrolments in the month<br>
ii. <b>total_updates</b>: Total demographic and biometric updates in the month<br>
iii. <b>total_activity</b>: Sum of enrolments and updates<br>
iv. <b>Update-to-Enrolment Ratio</b>: Aadhaar Infrastructure Stress Index<br>
v. <b>update_dependency_ratio</b>: Share of activity devoted to updates<br>
vi. <b>is_active</b>: Indicator of whether any Aadhaar activity occurred in the month
</small>




In [17]:
monthly_base.head()

,state,district,month,age_0_5,age_5_17,age_18_greater,demo_age_5_17,demo_age_17_,bio_age_5_17,bio_age_17_,new_enrolments,total_updates,total_activity,Update-to-Enrolment Ratio,update_dependency_ratio,is_active
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-03-01,0.0,0.0,0.0,0.0,0.0,16.0,193.0,0.0,209.0,209.0,209.0,0.995238,1
1,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-04-01,0.0,0.0,0.0,0.0,0.0,17.0,167.0,0.0,184.0,184.0,184.0,0.994595,1
2,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-05-01,0.0,0.0,0.0,0.0,0.0,22.0,158.0,0.0,180.0,180.0,180.0,0.994475,1
3,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-06-01,0.0,0.0,0.0,0.0,0.0,11.0,129.0,0.0,140.0,140.0,140.0,0.992908,1
4,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-07-01,0.0,0.0,0.0,0.0,0.0,20.0,239.0,0.0,259.0,259.0,259.0,0.996154,1


##### 9.2. District-Level Dormancy Index Calculation

To quantify how consistently Aadhaar infrastructure is used over time, dormancy is computed using the following formulas.

**Formulas:**

$$
\small
\text{Percentage of Active Months} = \frac{\text{Active Months}}{\text{Total Months}}
$$

$$
\small
\text{Dormancy Index} = 1 - \text{Percentage of Active Months}
$$

**Why these formulas are used:**
- The percentage of active months measures how frequently Aadhaar services are utilized in a district.
- The Dormancy Index converts this into an inactivity measure, where higher values indicate more persistent infrastructure dormancy.


In [18]:
district_dormancy_summary = (
    monthly_base
    .groupby(["state", "district"], as_index=False)
    .agg(
        active_months=("is_active", "sum"),
        total_months=("is_active", "count")
    )
)

district_dormancy_summary["pct_active_months"] = (
    district_dormancy_summary["active_months"] /
    district_dormancy_summary["total_months"]
)

district_dormancy_summary["dormancy_index"] = (
    1 - district_dormancy_summary["pct_active_months"]
)


##### 9.3. Preview of District Level Dormancy Metrics

<small>
The table shows district-level dormancy measures derived from monthly Aadhaar activity.
</small>

<small>

i. <b>active_months</b>: Number of months with recorded Aadhaar activity<br>
ii. <b>total_months</b>: Total number of observed months<br>
iii. <b>pct_active_months</b>: Proportion of months with Aadhaar activity<br>
iv. <b>dormancy_index</b>: Measure of infrastructure dormancy, where higher values indicate lower usage
</small>


In [19]:
district_dormancy_summary.head()

,state,district,active_months,total_months,pct_active_months,dormancy_index
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,9,10,0.9,0.1
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,3,10,0.3,0.7
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,4,10,0.4,0.6
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,9,10,0.9,0.1
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,9,10,0.9,0.1


##### 9.4. Longest Dormant Streak Computation

This step computes the longest continuous period of inactivity for each district by identifying the maximum number of consecutive months with no Aadhaar activity.

The longest dormant streak is calculated from the monthly activity indicator and merged with the district-level dormancy summary.


In [20]:
def longest_dormant_streak(x):
    max_streak = streak = 0
    for v in x:
        if v == 0:
            streak += 1
            max_streak = max(max_streak, streak)
        else:
            streak = 0
    return max_streak

dormant_streaks = (
    monthly_base
    .sort_values("month")
    .groupby(["state", "district"])["is_active"]
    .apply(longest_dormant_streak)
    .reset_index(name="max_dormant_streak")
)

district_dormancy_summary = district_dormancy_summary.merge(
    dormant_streaks,
    on=["state", "district"],
    how="left"
)


##### 9.5. Preview of District-Level Dormancy Summary

<small>
The table presents district-level dormancy metrics derived from monthly Aadhaar activity, including overall inactivity levels and the longest continuous dormant period observed.
</small>

<small>

i. <b>pct_active_months</b>: Proportion of months with recorded Aadhaar activity<br>
ii. <b>dormancy_index</b>: Measure of infrastructure dormancy, where higher values indicate lower utilization<br>
iii. <b>max_dormant_streak</b>: Longest continuous sequence of months with no Aadhaar activity
</small>


In [21]:
district_dormancy_summary

,state,district,active_months,total_months,pct_active_months,dormancy_index,max_dormant_streak
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,9,10,0.9,0.1,1
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,3,10,0.3,0.7,6
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,4,10,0.4,0.6,6
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,9,10,0.9,0.1,1
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,9,10,0.9,0.1,1
...,...,...,...,...,...,...,...
1089,WEST BENGAL,WEST MEDINIPUR,4,10,0.4,0.6,6
1090,WEST BENGAL,WEST MIDNAPORE,9,10,0.9,0.1,1
1091,WEST BENGLI,HOOGHLY,2,10,0.2,0.8,8
1092,WESTBENGAL,HOOGHLY,4,10,0.4,0.6,6


In [22]:
district_dormancy_summary.to_csv("data/district_summary/district_dormancy_summary.csv")

##### 10. Biometric Decay Score (BDS)

The Biometric Decay Score (BDS) captures the extent to which Aadhaar activity among adults is dominated by biometric updates, indicating potential biometric degradation or capture quality issues.

**Formula:**

$$
\small
\text{Biometric Decay Score (BDS)} = \frac{\text{Adult Biometric Updates}}{\text{Total Adult Updates}}
$$

**Components:**
- **Adult Biometric Updates** represent biometric update transactions for the adult population.
- **Total Adult Updates** represent the total number of demographic and biometric updates for adults.

**Why this metric is computed:**
- Indicates biometric degradation or poor capture quality over time.
- High values suggest frequent fingerprint or iris failures among adults.

**Policy relevance:**
- Informs hardware refresh decisions.
- Supports review of biometric capture modalities and update protocols.

The Biometric Decay Score is computed independently and is not combined with the other operational indices.




In [23]:
monthly_base["adult_total_updates"] = (
    monthly_base["bio_age_17_"] +
    monthly_base["demo_age_17_"]
)

monthly_base["bds"] = (
    monthly_base["bio_age_17_"] /
    (monthly_base["adult_total_updates"] + 1)
)


##### 10.1. District-Level Aggregation of Biometric Decay Score

Monthly Biometric Decay Score values are aggregated at the district level to summarize typical and peak levels of biometric decay.


In [24]:
district_bds_summary = (
    monthly_base
    .groupby(["state", "district"], as_index=False)
    .agg(
        mean_bds=("bds", "mean"),
        max_bds=("bds", "max")
    )
)

##### 10.2. Preview of District-Level Biometric Decay Score

<small>
The table presents district-level summaries of the Biometric Decay Score, showing average and peak levels of biometric update dominance among adult Aadhaar updates.
</small>

<small>

i. <b>mean_bds</b>: Average Biometric Decay Score for the district<br>
ii. <b>max_bds</b>: Highest observed Biometric Decay Score
</small>


In [25]:
district_bds_summary.head()

,state,district,mean_bds,max_bds
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,0.711003,0.995833
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,0.050000,0.500000
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,0.205137,0.580645
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,0.603048,0.993243
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,0.547936,0.997797


In [26]:
district_bds_summary.to_csv("data/district_summary/district_bds_summary.csv", index=False)

##### 11. Child Compliance Index (CCI)

The Child Compliance Index (CCI) measures the extent to which Aadhaar activity for children reflects compliance with mandatory biometric update requirements at key age milestones.

**Formula:**

$$
\small
\text{Child Compliance Index (CCI)} = \frac{\text{Mandatory Child Biometric Updates (ages 5–17)}}{\text{Total Child Activity}}
$$

**Components:**
- **Mandatory Child Biometric Updates (5–17)** represent biometric update transactions for children subject to mandatory updates.
- **Total Child Activity** represents all Aadhaar enrolment and update activity for children.

**Why this metric is computed:**
- Measures compliance with mandatory biometric updates at the 5- and 15-year milestones.
- Low values indicate awareness gaps or barriers to access.

**Policy relevance:**
- Informs school-linked enrolment and update campaigns.
- Supports targeted parental outreach initiatives.


In [27]:
monthly_base["child_total_activity"] = (
    monthly_base["age_5_17"] +
    monthly_base["demo_age_5_17"] +
    monthly_base["bio_age_5_17"]
)

monthly_base["cci"] = (
    monthly_base["bio_age_5_17"] /
    (monthly_base["child_total_activity"] + 1)
)


##### 11.1 District-Level Aggregation of Child Compliance Index

Monthly Child Compliance Index values are aggregated at the district level to summarize typical compliance levels and the lowest observed compliance over time.


In [28]:
district_cci_summary = (
    monthly_base
    .groupby(["state", "district"], as_index=False)
    .agg(
        mean_cci=("cci", "mean"),
        min_cci=("cci", "min")
    )
)


##### 11.2. Preview of District-Level Child Compliance Index

<small>
The table presents district-level summaries of the Child Compliance Index, reflecting typical compliance levels and the lowest observed compliance across months.
</small>

<small>

i. <b>mean_cci</b>: Average Child Compliance Index for the district<br>
ii. <b>min_cci</b>: Lowest observed Child Compliance Index for the district
</small>


In [29]:
district_cci_summary.head(10)

,state,district,mean_cci,min_cci
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,0.848777,0.0
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,0.050000,0.0
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,0.342552,0.0
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,0.831662,0.0
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,0.871154,0.0
5,ANDAMAN AND NICOBAR ISLANDS,SOUTH ANDAMAN,0.845995,0.0
6,ANDHRA PRADESH,ADILABAD,0.754645,0.0
7,ANDHRA PRADESH,ALLURI SITHARAMA RAJU,0.750180,0.0
8,ANDHRA PRADESH,ANAKAPALLI,0.597806,0.0
9,ANDHRA PRADESH,ANANTAPUR,0.731795,0.0


In [30]:
district_cci_summary.to_csv("data/district_summary/district_cci_summary.csv", index=False)

In [31]:
monthly_base.to_csv("data/monthly_metrics.csv", index=False)

### Conclusion

This notebook engineered a set of district-level indices and indicators from Aadhaar enrolment and update data, capturing infrastructure stress, update dependency, dormancy, biometric decay (BDS), and child compliance (CCI).

These outputs serve as **structured inputs** for the next stage of analysis, where they are combined to construct the **Digital Maturity Gradient** and assess district-level system behavior.


### Building the Digital Maturity feature table

In [32]:
digital_maturity_df = (
    district_stress_summary
    .merge(district_dependency_summary, on=["state", "district"], how="left")
    .merge(district_dormancy_summary, on=["state", "district"], how="left")
)


In [33]:
from scipy.stats import linregress

def compute_growth(group):
    x = np.arange(len(group))
    y = group["total_activity"].values
    if len(y) < 2:
        return 0.0
    return linregress(x, y).slope

growth_df = (
    monthly_base
    .sort_values("month")
    .groupby(["state", "district"])
    .apply(compute_growth)
    .reset_index(name="activity_growth_slope")
)

digital_maturity_df = digital_maturity_df.merge(
    growth_df, on=["state", "district"], how="left"
)


/tmp/ipykernel_98332/4017408124.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_growth)


In [34]:
digital_maturity_df

,state,district,mean_update_to_enrolment_ratio,max_update_to_enrolment_ratio,update_to_enrolment_ratio_volatility,active_months_x,mean_update_dependency,max_update_dependency,dependency_volatility,active_months_y,total_months,pct_active_months,dormancy_index,max_dormant_streak,activity_growth_slope
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,106.583631,259.000000,97.416098,10,0.879568,0.996154,0.309773,9,10,0.9,0.1,1,42.703030
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,0.550000,4.000000,1.257201,10,0.163333,0.800000,0.285644,3,10,0.3,0.7,6,0.260606
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,6.728472,21.666667,9.163387,10,0.376071,0.955882,0.485612,4,10,0.4,0.6,6,26.375758
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,149.532449,671.000000,207.028878,10,0.873744,0.998512,0.308890,9,10,0.9,0.1,1,-9.145455
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,501.677162,1437.000000,563.351927,10,0.884397,0.999305,0.311274,9,10,0.9,0.1,1,-12.933333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1089,WEST BENGAL,WEST MEDINIPUR,1.466667,7.000000,2.515139,10,0.277500,0.875000,0.379247,4,10,0.4,0.6,6,0.606061
1090,WEST BENGAL,WEST MIDNAPORE,976.959575,2762.000000,1074.952371,10,0.878452,0.999638,0.310034,9,10,0.9,0.1,1,1274.193939
1091,WEST BENGLI,HOOGHLY,0.300000,2.000000,0.674949,10,0.116667,0.666667,0.249072,2,10,0.2,0.8,8,0.151515
1092,WESTBENGAL,HOOGHLY,6.866667,34.000000,11.410630,10,0.368235,0.971429,0.476377,4,10,0.4,0.6,6,7.400000


In [35]:
digital_maturity_features = digital_maturity_df[
    [
        "state",
        "district",
        "mean_update_to_enrolment_ratio",
        "mean_update_dependency",
        "update_to_enrolment_ratio_volatility",
        "dependency_volatility",
        "pct_active_months",
        "activity_growth_slope"
    ]
]

In [36]:
digital_maturity_features.head()

,state,district,mean_update_to_enrolment_ratio,mean_update_dependency,update_to_enrolment_ratio_volatility,dependency_volatility,pct_active_months,activity_growth_slope
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,106.583631,0.879568,97.416098,0.309773,0.9,42.703030
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,0.550000,0.163333,1.257201,0.285644,0.3,0.260606
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,6.728472,0.376071,9.163387,0.485612,0.4,26.375758
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,149.532449,0.873744,207.028878,0.308890,0.9,-9.145455
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,501.677162,0.884397,563.351927,0.311274,0.9,-12.933333


In [37]:
digital_maturity_features.isna().sum()


state                                   0
district                                0
mean_update_to_enrolment_ratio          0
mean_update_dependency                  0
update_to_enrolment_ratio_volatility    0
dependency_volatility                   0
pct_active_months                       0
activity_growth_slope                   0
dtype: int64

In [38]:
digital_maturity_features.to_csv("data/digital_maturity_features.csv", index=False)

In [39]:
monthly_base.head()

,state,district,month,age_0_5,age_5_17,age_18_greater,demo_age_5_17,demo_age_17_,bio_age_5_17,bio_age_17_,new_enrolments,total_updates,total_activity,Update-to-Enrolment Ratio,update_dependency_ratio,is_active,adult_total_updates,bds,child_total_activity,cci
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-03-01,0.0,0.0,0.0,0.0,0.0,16.0,193.0,0.0,209.0,209.0,209.0,0.995238,1,193.0,0.994845,16.0,0.941176
1,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-04-01,0.0,0.0,0.0,0.0,0.0,17.0,167.0,0.0,184.0,184.0,184.0,0.994595,1,167.0,0.994048,17.0,0.944444
2,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-05-01,0.0,0.0,0.0,0.0,0.0,22.0,158.0,0.0,180.0,180.0,180.0,0.994475,1,158.0,0.993711,22.0,0.956522
3,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-06-01,0.0,0.0,0.0,0.0,0.0,11.0,129.0,0.0,140.0,140.0,140.0,0.992908,1,129.0,0.992308,11.0,0.916667
4,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-07-01,0.0,0.0,0.0,0.0,0.0,20.0,239.0,0.0,259.0,259.0,259.0,0.996154,1,239.0,0.995833,20.0,0.952381


### Mean_monthly activity

In [40]:
activity_heatmap_df = (
    monthly_base
    .groupby(["state", "district"], as_index=False)
    .agg(
        mean_monthly_activity=("total_activity", "mean")
    )
)


In [41]:
activity_heatmap_df.head()

,state,district,mean_monthly_activity
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,284.0
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,0.7
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,71.0
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,267.3
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,852.4


In [42]:
activity_heatmap_df.to_csv("data/mean_monthly_activity.csv", index=False)

In [43]:
monthly_base.head()

,state,district,month,age_0_5,age_5_17,age_18_greater,demo_age_5_17,demo_age_17_,bio_age_5_17,bio_age_17_,new_enrolments,total_updates,total_activity,Update-to-Enrolment Ratio,update_dependency_ratio,is_active,adult_total_updates,bds,child_total_activity,cci
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-03-01,0.0,0.0,0.0,0.0,0.0,16.0,193.0,0.0,209.0,209.0,209.0,0.995238,1,193.0,0.994845,16.0,0.941176
1,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-04-01,0.0,0.0,0.0,0.0,0.0,17.0,167.0,0.0,184.0,184.0,184.0,0.994595,1,167.0,0.994048,17.0,0.944444
2,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-05-01,0.0,0.0,0.0,0.0,0.0,22.0,158.0,0.0,180.0,180.0,180.0,0.994475,1,158.0,0.993711,22.0,0.956522
3,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-06-01,0.0,0.0,0.0,0.0,0.0,11.0,129.0,0.0,140.0,140.0,140.0,0.992908,1,129.0,0.992308,11.0,0.916667
4,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-07-01,0.0,0.0,0.0,0.0,0.0,20.0,239.0,0.0,259.0,259.0,259.0,0.996154,1,239.0,0.995833,20.0,0.952381


### Capped the stess index for visualization

In [44]:
cap = monthly_base["Update-to-Enrolment Ratio"].quantile(0.99)
monthly_base["Update-to-Enrolment Ratio_capped"] = monthly_base["Update-to-Enrolment Ratio"].clip(upper=cap)

In [45]:
monthly_base.head()

,state,district,month,age_0_5,age_5_17,age_18_greater,demo_age_5_17,demo_age_17_,bio_age_5_17,bio_age_17_,...,total_updates,total_activity,Update-to-Enrolment Ratio,update_dependency_ratio,is_active,adult_total_updates,bds,child_total_activity,cci,Update-to-Enrolment Ratio_capped
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-03-01,0.0,0.0,0.0,0.0,0.0,16.0,193.0,...,209.0,209.0,209.0,0.995238,1,193.0,0.994845,16.0,0.941176,209.0
1,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-04-01,0.0,0.0,0.0,0.0,0.0,17.0,167.0,...,184.0,184.0,184.0,0.994595,1,167.0,0.994048,17.0,0.944444,184.0
2,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-05-01,0.0,0.0,0.0,0.0,0.0,22.0,158.0,...,180.0,180.0,180.0,0.994475,1,158.0,0.993711,22.0,0.956522,180.0
3,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-06-01,0.0,0.0,0.0,0.0,0.0,11.0,129.0,...,140.0,140.0,140.0,0.992908,1,129.0,0.992308,11.0,0.916667,140.0
4,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-07-01,0.0,0.0,0.0,0.0,0.0,20.0,239.0,...,259.0,259.0,259.0,0.996154,1,239.0,0.995833,20.0,0.952381,259.0


In [46]:
monthly_base.to_csv("data/monthly_metrics.csv", index=False)